HW is sent to Discord Channel, due at start of next class.

## Pandas Homework — 10 Exercises

Use the `data.csv` employee dataset from class. Unless specifically instructed otherwise, solve each problem using Pandas rather than loops over individual rows.

### Challenge constraint

For **questions 2, 5, 7, 9, and 10**, do not use a Python `for` loop to process employees individually. The objective is to solve the problem using DataFrame operations such as `groupby()`, `transform()`, `rank()`, `agg()`, `merge()`, boolean indexing, and related Pandas functionality.

In [2]:
# Setup
from pathlib import Path
import csv

import pandas as pd

# If the assertion cannot find data.csv, run Path.cwd() and check where the notebook kernel started. 
# Your notebook and CSV are in the same Week 2 directory, but relative paths are resolved against the kernel’s working directory—not necessarily the notebook’s displayed location.
DATA_PATH = Path("data.csv")

# check we used the corret dir to access the csv
print("Working directory:", Path.cwd())

# safely check the path to the csv
assert DATA_PATH.exists(), f"CSV not found at {DATA_PATH.resolve()}"

# Pure Python: DictReader initially returns each field as a string.
with DATA_PATH.open(mode="r", encoding="utf-8", newline="") as csv_file:
    reader = csv.DictReader(csv_file)
    raw_employees = list(reader)

# how many rows
print("Pure Python records:", len(raw_employees))

# print schema?
print(raw_employees[0])

# pandas: load the columns using an explicit schema.
# not required, but safer
df = pd.read_csv(
    DATA_PATH,
    usecols=[
        "employee_id",
        "name",
        "department",
        "salary",
        "years_experience",
        "active",
    ],
    dtype={
        "employee_id": "Int64",
        "name": "string",
        "department": "string",
        "salary": "Float64",
        "years_experience": "Int64",
        "active": "boolean",
    },
    true_values=["true", "TRUE"],
    false_values=["false", "FALSE"],
)

# shape = ?
print("DataFrame shape:", df.shape)
display(df.head())

# info = ?
df.info()

Working directory: /Users/richardwilkerson/VSCodeProjects/BigDataExample/docs/Training/Week2
Pure Python records: 100
{'employee_id': '1', 'name': 'Alice Johnson', 'department': 'Engineering', 'salary': '92000', 'years_experience': '6', 'active': 'true'}
DataFrame shape: (100, 6)


,employee_id,name,department,salary,years_experience,active
0,1,Alice Johnson,Engineering,92000.0,6,True
1,2,Bob Smith,Sales,68000.0,4,True
2,3,Carol Williams,Engineering,105000.0,9,True
3,4,David Brown,Marketing,62000.0,3,True
4,5,Emma Davis,Finance,88000.0,7,True


<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   employee_id       100 non-null    Int64  
 1   name              100 non-null    string 
 2   department        100 non-null    string 
 3   salary            100 non-null    Float64
 4   years_experience  100 non-null    Int64  
 5   active            100 non-null    boolean
dtypes: Float64(1), Int64(2), boolean(1), string(2)
memory usage: 4.5 KB


1. **Department statistics**
   Produce a DataFrame containing one row per department with:

   * Number of employees
   * Average salary
   * Minimum salary
   * Maximum salary
   * Average years of experience

   Sort the result from highest to lowest average salary.

In [ ]:

# this question teaches how agg works like a reducer to reduce rows based on columns

# Note, only the column salary is passed to named functions like this
def diffMinMax(salaries):
    return salaries.max() - salaries.min()

# get each unique department
department = (
    # Group employees that share the same department.
    # as_index=False keeps "department" as a regular output column
    # instead of making it the DataFrame index.
    df.groupby("department", as_index=False)
    # agg is used to create the table columns you want
    .agg(
        # Named aggregation syntax:
        # output_column_name=(source_column_name, aggregation)        
        employee_count = ("employee_id", "count"),
        avg_salary = ("salary", "mean"),
        min_salary = ("salary", min),
        max_salary = ("salary", max),
        avg_exp = ("years_experience", "mean"),
        diff = ("salary", diffMinMax),
        total_salary = ("salary", sum),
        total_exp = ("years_experience", sum)
    )
    # can also use assign to take calculations you've already done to create a column
    .assign(
        total_salary_per_exp = lambda result:
            result["total_salary"]/result["total_exp"]
    )
    # Place the department with the highest average salary first.
    .sort_values("avg_salary", ascending=False)
  )
# display() provides rich table formatting inside a notebook.
# print() also works, but produces a plain-text representation.
display(department)

# agg key words:
  # "size"    -> number of rows, including rows containing missing values
  # "count"   -> number of non-null values in the selected column
  # "nunique" -> number of distinct non-null values
  # "sum"     -> total
  # "mean"    -> arithmetic average
  # "median"  -> middle value
  # "min"     -> smallest value
  # "max"     -> largest value
  # "std"     -> standard deviation
  # "var"     -> variance
  # "first"   -> first non-null value in each group
  # "last"    -> last non-null value in each group
  # can use lambda or named function as well



2. **Above-department-average employees**
   Find every employee whose salary is **greater than the average salary of their own department**. Display:

   ```text
   name | department | salary | department_average
   ```

   Sort by department and then salary descending.

In [ ]:
# this question teaches how transform() works, when you want to keep every row, but transform values
# calculates a group value and returns it aligned with every original employee row.
# transform is like a reducer, it takes all rows and creates a single value

# Note, you can use columns defined sequentially. 
# loc can call "department_average" because it was declared before it, but it cannot use something if it came after.

# question, what's a good way of thinking about when to use () vs [] here? like we have groupby("department")["salary"] 

# A useful rule for () versus [] is:
# - () calls a function or method.
# - [] selects something from an object or describes an indexing operation.
# - [...] can also construct a Python list, depending on its position.

above_department_average = (
    df.assign(
        # Calculate each department's average and place that value
        # beside every employee belonging to that department.
        department_average=(
            # displays all 100 rows, but organized by department, returns DataFrameGroupBy
            df.groupby("department") 
            # within each department group ("Engineer", "HR", etc.), select the salary column for all rows. retruns SeriesGroupBy
            # the GroupBy object already exists. ["salary"] selects a column from it.
            ["salary"]
            # transform reduces those values to a single value based on the method used ("mean"). returns Series
            .transform("mean")
        )
    )

    # .loc[rows_to_keep, columns_to_keep]
    # First expression selects rows (employees["salary"] > employees["department_average"]); the list selects and orders columns.
    # .loc is an indexing property rather than a method, so it uses [], not ().
    # if you want to select every row, use ":" -> not True
        # In .loc[True, columns], True does not mean “include all rows.” Because it is one scalar value, 
        # pandas treats it like a row label and tries to find an index row literally labeled True.
    # you can use index numbers if you want, [1, 4, 7, etc.]

    .loc[
        lambda employees:
            employees["salary"] > employees["department_average"], 
            # define columns you want to show
            ["name", "department", "salary", "department_average"],
    ]

    # Sort department ascending, then salary descending within departments.
    # the order determines the priority when sorting
    # ascending defaults to true if missing
    .sort_values(
        ["department", "salary"],
        ascending=[True, False],
    )

    # Replace the filtered original index 
    # Suppose the original DataFrame has these index labels:
        #   index | name
        #   0     | Alice
        #   1     | Bob
        #   2     | Carol
        #   3     | David
        #   4     | Elena
    # If only Bob and Elena earn above their department averages, the filtered result retains their original labels:
        #   index | name
        #   1     | Bob
        #   4     | Elena
    .reset_index(drop=True)
)

display(above_department_average)
# above_department_average.head

3. **Salary bands**
   Add a new column called `salary_band` according to these rules:

   ```text
   Under 70,000        -> "Low"
   70,000–89,999       -> "Medium"
   90,000–109,999      -> "High"
   110,000 and above   -> "Very High"
   ```

   Then determine how many employees from each department fall into each salary band.

In [ ]:
def salaryBand(salary):
    if salary < 70000: return "Low"
    elif salary < 90000: return "Medium"
    elif salary < 110000: return "High"
    elif salary >= 110000: return "Very High"

employees = (
    # use assign to create a new column
    df.assign(
        # use map to iterate through each row and set the value for salary_band
        salary_band = lambda current: current["salary"].map(salaryBand)

        # pd also has cut; designed to turn continuous numeric values into categorical ranges:
        # salary_band=lambda current: pd.cut(
        #       current["salary"],
        #       bins=[
        #           -float("inf"),
        #           70_000,
        #           90_000,
        #           110_000,
        #           float("inf"),
        #       ],
        #       labels=[
        #           "Low",
        #           "Medium",
        #           "High",
        #           "Very High",
        #       ],
        ## makes the intervals include their left boundary and exclude their right boundary:
        #       right=False,
        #   )
    )
)
salary_band_counts = (
    # Note, we are using employees from above, not df. 
    employees
    # Note, groupby always includes those columns
        .groupby(
            ["department", "salary_band"],
            observed = True
        )
        .size()
        .rename("band_count")
        .reset_index()
)

display(employees)
display(salary_band_counts)


,employee_id,name,department,salary,years_experience,active,salary_band
0,1,Alice Johnson,Engineering,92000.0,6,True,High
1,2,Bob Smith,Sales,68000.0,4,True,Low
2,3,Carol Williams,Engineering,105000.0,9,True,High
3,4,David Brown,Marketing,62000.0,3,True,Low
4,5,Emma Davis,Finance,88000.0,7,True,Medium
...,...,...,...,...,...,...,...
95,96,Sara Griffin,Engineering,114000.0,10,True,Very High
96,97,Trevor Diaz,Sales,86000.0,7,False,Medium
97,98,Violet Hayes,Marketing,80000.0,6,True,Medium
98,99,Wyatt Myers,Finance,100000.0,9,True,High


,department,salary_band,band_count
0,Engineering,High,17
1,Engineering,Medium,8
2,Engineering,Very High,6
3,Finance,High,12
4,Finance,Medium,8
5,HR,Low,7
6,HR,Medium,7
7,Marketing,Low,3
8,Marketing,Medium,12
9,Sales,Low,3


4. **Experienced but underpaid**
   Find employees who have **at least 7 years of experience** but earn **less than $90,000**. Determine which department has the largest number of these employees.

In [92]:
underpaid = (
    df.loc[
        lambda employees: (
            (employees["salary"] < 90000) 
            & (employees["years_experience"] >= 7)
        ),
        :
    ]
)

mostUnderpaid = (
    underpaid
    .groupby("department")
    .size()
    .rename("employee_count")
    .reset_index()
    .sort_values("employee_count", ascending=False)
)



display(underpaid)
display(mostUnderpaid)

,employee_id,name,department,salary,years_experience,active
4,5,Emma Davis,Finance,88000.0,7,True
21,22,Victor Lewis,Sales,83000.0,7,True
47,48,Vince Collins,Finance,89000.0,7,True
76,77,Yosef Barnes,Sales,85000.0,7,True
91,92,Nina Gonzales,Sales,84000.0,7,True
96,97,Trevor Diaz,Sales,86000.0,7,False


,department,employee_count
1,Sales,4
0,Finance,2


5. **Department ranking**
   Add a column called `salary_rank` that ranks employees by salary **within their own department**, where the highest-paid employee in each department receives rank 1. Display the three highest-paid employees from each department.

In [ ]:
# This helps practice pythons sequential ordering
# rank employees by department
# sort those values based on department and salary_rank
# now we want to take from each department -> groupby
# apply head() to each group
# select the columns we want to show
ranking = (
    df.assign(
        salary_rank = lambda employees: (
            employees
            .groupby("department")["salary"]
            # built in ordering with number
            .rank(
                # controls ties, 
                # min = both get same rank, and next rank is skipped, 1,1,3
                # dense = both get same rank, but next is not skipped, 1,1,2
                # first = first to be seen gets 1, 1,2,3
                method = "min",
                # start counting from 1
                ascending = False,
            )
            .astype("Int64")
        )
    )
    .sort_values(
        ["department", "salary_rank"],
        ascending=[True, True]
    )
    # group_keys = ?
    .groupby("department", group_keys=False)
    .head(3)
    .loc[
        :, ["employee_id", "name", "department", "salary"]
    ]
    .reset_index(drop=True)

)

display(ranking)

,employee_id,name,department,salary
0,43,Queen Phillips,Engineering,120000.0
1,76,Xena Wood,Engineering,118000.0
2,23,Wendy Lee,Engineering,115000.0
3,19,Samuel Robinson,Finance,102000.0
4,99,Wyatt Myers,Finance,100000.0
5,79,Adam Henderson,Finance,99000.0
6,95,Roger Russell,HR,75000.0
7,75,Walter Bennett,HR,74000.0
8,61,Ivy Richardson,HR,73000.0
9,98,Violet Hayes,Marketing,80000.0


6. **Create and perform a join**
   Create a second DataFrame in your Python program:

   ```python
   department_info = pd.DataFrame({
       "department": [
           "Engineering",
           "Sales",
           "Marketing",
           "Finance",
           "HR"
       ],
       "manager": [
           "Sarah Connor",
           "Michael Scott",
           "Don Draper",
           "Bruce Wayne",
           "Leslie Knope"
       ],
       "location": [
           "New York",
           "Chicago",
           "Los Angeles",
           "New York",
           "Chicago"
       ]
   })
   ```

   Join this DataFrame with the employee data so that every employee record also contains their manager and office location. Then calculate the average salary for each office location.

In [114]:
department_info = pd.DataFrame({
    "department": [
        "Engineering",
        "Sales",
        "Marketing",
        "Finance",
        "HR"
    ],
    "manager": [
        "Sarah Connor",
        "Michael Scott",
        "Don Draper",
        "Bruce Wayne",
        "Leslie Knope"
    ],
    "location": [
        "New York",
        "Chicago",
        "Los Angeles",
        "New York",
        "Chicago"
    ]
})

joined = (
    df.merge(
        department_info,
        # Match rows using the department column in both DataFrames.
        on="department",

        # Keep every employee, even if their department does not
        # have a matching department_info row.
        how="left",

        # Assert that many employees may match one department,
        # but department_info must not contain duplicate departments.
        # The validate="many_to_one" check is especially valuable in data pipelines. 
        # If department_info accidentally contains two Engineering rows, the join could duplicate every Engineering employee. 
        # Validation makes pandas raise an error instead.
        validate="many_to_one",
    )
)

avg_sal = (
    joined.groupby("location", as_index=False)
    .agg(
        average_salary = ("salary", "mean"),
    )
)

display(joined)
display(avg_sal)

,employee_id,name,department,salary,years_experience,active,manager,location
0,1,Alice Johnson,Engineering,92000.0,6,True,Sarah Connor,New York
1,2,Bob Smith,Sales,68000.0,4,True,Michael Scott,Chicago
2,3,Carol Williams,Engineering,105000.0,9,True,Sarah Connor,New York
3,4,David Brown,Marketing,62000.0,3,True,Don Draper,Los Angeles
4,5,Emma Davis,Finance,88000.0,7,True,Bruce Wayne,New York
...,...,...,...,...,...,...,...,...
95,96,Sara Griffin,Engineering,114000.0,10,True,Sarah Connor,New York
96,97,Trevor Diaz,Sales,86000.0,7,False,Michael Scott,Chicago
97,98,Violet Hayes,Marketing,80000.0,6,True,Don Draper,Los Angeles
98,99,Wyatt Myers,Finance,100000.0,9,True,Bruce Wayne,New York


,location,average_salary
0,Chicago,74147.058824
1,Los Angeles,73133.333333
2,New York,95745.098039


7. **Detect salary outliers**
   For each department, calculate its mean salary and standard deviation. Find employees whose salary is **more than one standard deviation above their department's mean**. Do not hard-code the department averages or standard deviations.

In [128]:
dept_sal_std = (
    df.groupby("department", as_index=False)
    .agg(
        average_salary = ("salary", "mean"),
        standard_devi = ("salary", "std"),
    )
)
emp_above_std = (
    df.merge(
        dept_sal_std,
        on = "department",
        how = "left",
        validate = "many_to_one",
    )
    .loc[
        lambda employees: (
            employees["salary"] > (
                employees["salary"] + employees["standard_devi"]
            )
        ), :
    ]
    .reset_index(drop=True)
)

display(dept_sal_std)
display(emp_above_std)

,department,average_salary,standard_devi
0,Engineering,98677.419355,11634.394689
1,Finance,91200.0,6329.546421
2,HR,69642.857143,3387.922959
3,Marketing,73133.333333,4748.934718
4,Sales,77300.0,5685.623606


,employee_id,name,department,salary,years_experience,active,average_salary,standard_devi


8. **Simulate a compensation adjustment**
   The company decides to implement the following raises:

   ```text
   Salary < $70,000             -> 8% raise
   Salary $70,000–$89,999       -> 5% raise
   Salary $90,000 or greater    -> 3% raise
   ```

   Add a `new_salary` column without changing the original `salary` column. Then calculate:

   * Original total payroll
   * New total payroll
   * Total cost of the raises
   * Percentage increase in payroll

In [145]:
def compAdjustment(salary) :
    if salary < 70000: return (salary * .08) + salary
    elif salary < 90000: return (salary * .05) + salary
    else: return (salary * .03) + salary

newSalary = (
    df.assign(
        new_salary = lambda employees:
        employees["salary"].map(compAdjustment)
    )
)

ori_total_pay = newSalary["salary"].sum()
new_total_pay = newSalary["new_salary"].sum()
raise_cost = new_total_pay - ori_total_pay
per_inc_pay = (raise_cost/ori_total_pay) * 100

payrollUpdates = pd.DataFrame({
    "ori_total_pay": [ori_total_pay],
    "new_total_pay": [new_total_pay],
    "raise_cost": [raise_cost],
    "per_inc_pay": [per_inc_pay]
})

display(newSalary)
display(payrollUpdates)


,employee_id,name,department,salary,years_experience,active,new_salary
0,1,Alice Johnson,Engineering,92000.0,6,True,94760.0
1,2,Bob Smith,Sales,68000.0,4,True,73440.0
2,3,Carol Williams,Engineering,105000.0,9,True,108150.0
3,4,David Brown,Marketing,62000.0,3,True,66960.0
4,5,Emma Davis,Finance,88000.0,7,True,92400.0
...,...,...,...,...,...,...,...
95,96,Sara Griffin,Engineering,114000.0,10,True,117420.0
96,97,Trevor Diaz,Sales,86000.0,7,False,90300.0
97,98,Violet Hayes,Marketing,80000.0,6,True,84000.0
98,99,Wyatt Myers,Finance,100000.0,9,True,103000.0


,ori_total_pay,new_total_pay,raise_cost,per_inc_pay
0,8501000.0,8881550.0,380550.0,4.476532


9. **Experience vs. compensation**
   Create experience groups:

   ```text
   0–4 years    -> Junior
   5–7 years    -> Mid
   8+ years     -> Senior
   ```,

   Calculate the average salary for each experience group **within each department**. Determine which department has the largest salary difference between its Junior and Senior employees. Handle departments that don't have employees in both groups appropriately.

In [ ]:
def expGroup(exp):
    if exp < 5: return "Junior"
    elif exp < 8: return "Mid"
    else: return "Senior"

expVsComp = (
    df.assign(
        exp_group = lambda employees: employees["years_experience"].map(expGroup)
    )
    .groupby(["department", "exp_group"], as_index=False)
    .agg(
        avg_sal = ("salary", "mean")
    )
)
salaryDiff = (
    expVsComp
    # pivot allows us to redefine the table, please explain how this works in more detail.
        # pivot() needs each department and experience-group combination to be unique. 
        # Otherwise, pandas would not know which value to place in a cell.
        # this is made possible by our previous call to agg avg_sal
    # index = What should identify each resulting row?
        # Each department becomes one row. Department moves from a regular column to the DataFrame’s index
    # columns = Which existing values should become new column names?
        # distinct values from exp_group become columns
    # values = What values should pandas place inside the newly arranged cells?
        # Pandas fills the cells using the average salaries
    .pivot(
        index="department",
        columns="exp_group",
        values="avg_sal",
    )
    # reindex: selects, creates, and orders the columns:
    # - Places the columns in the desired Junior, Mid, Senior order.
    # - Creates a missing column filled with NaN if the entire dataset has no employees in one group.
    # without reindex, a missing value for a column might get dropped -> later calls might be null -> breaks code
    .reindex(columns=["Junior", "Mid", "Senior"])
    # dropna removes departments that cannot be compared:
    # The subset argument means: Check only the Junior and Senior columns when deciding which rows to remove.
    # if either cannot be compared (NaN) -> drop
    .dropna(subset=["Junior", "Senior"])
    # now we can do our comparison without null 
    .assign(
        salary_diff = lambda department: (
            # easier calc once exp_group become columns
            department["Senior"] - department["Junior"]
        )
    )
    # sort high to low, 
    # Asc = True -> low to high
    # Asc = False -> high to low
    .sort_values(
        ["salary_diff"],
        ascending=[False] )
    # take only the top value
    .head(1)
    .reset_index()
)

display(expVsComp)
display(salaryDiff)

,department,exp_group,avg_sal
0,Engineering,Junior,76000.0
1,Engineering,Mid,89000.0
2,Engineering,Senior,107411.764706
3,Finance,Mid,87461.538462
4,Finance,Senior,98142.857143
5,HR,Junior,67166.666667
6,HR,Mid,71500.0
7,Marketing,Junior,66333.333333
8,Marketing,Mid,74833.333333
9,Sales,Junior,67666.666667


exp_group,department,Junior,Mid,Senior,salary_diff
0,Engineering,76000.0,89000.0,107411.764706,31411.764706


10. **Mini ETL challenge**
    Write a Pandas program that reads `data.csv` and produces a new file called:

    ```text
    department_report.csv
    ```

    The output must contain exactly one row per department and these columns:

    ```text
    department
    employee_count
    active_employee_count
    average_salary
    median_salary
    average_experience
    highest_salary
    highest_paid_employee
    ```

    Round monetary averages to two decimal places and sort the final file by `average_salary` descending.

In [177]:
departmentReport = (
    df
    .groupby("department", as_index=False)
    .agg(
        employee_count = ("employee_id", "count"),
        active_employee_count = ("active", lambda active: active.fillna(False).sum()),
        average_salary = ("salary", "mean"),
        median_salary = ("salary", "median"),
        average_experience = ("years_experience", "mean"),
        highest_salary = ("salary", max),
    )
)

highestPaidID = (
    df.groupby("department")["salary"].idxmax()
)
highestPaid = (
    df.loc[
        # true where the employee_id is found -> grab the department and name to merge later
        highestPaidID, ["department", "name"],
    ]
    .rename(columns = {"name": "highest_paid_employee"})
)

departmentReport = (
    departmentReport.merge(
        highestPaid,
        on="department",
        how="left",
        validate="one_to_one"
    )
    .round({
        "average_salary": 2,
        "median_salary": 2
    })
    # big -> small
    .sort_values(
        ["average_salary"],
        ascending = [False]
    )
)

display(departmentReport)

# index = False, 
departmentReport.to_csv(
    "department_report.csv", index=False
)

,department,employee_count,active_employee_count,average_salary,median_salary,average_experience,highest_salary,highest_paid_employee
0,Engineering,31,29,98677.42,98000.0,7.870968,120000.0,Queen Phillips
1,Finance,20,18,91200.0,91000.0,7.15,102000.0,Samuel Robinson
4,Sales,20,17,77300.0,78000.0,5.5,86000.0,Trevor Diaz
3,Marketing,15,14,73133.33,73000.0,5.133333,80000.0,Violet Hayes
2,HR,14,13,69642.86,69500.0,4.571429,75000.0,Roger Russell


In [ ]:
()